# Синтетические эксперименты по восстановлению матриц

Цель тетради — сравнить методы восстановления матриц малого ранга на контролируемых сценариях.
В каждом эксперименте меняется только один параметр: ранг, шум или тип пропусков.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
for candidate in (PROJECT_DIR, *PROJECT_DIR.parents):
    if (candidate / 'synthetic_api.py').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError('Не удалось найти корень проекта')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import synthetic_research_api as research

PROJECT_DIR


## 1. Базовый сценарий

Фиксируем матрицу 500 × 500 истинного ранга 3, случайные пропуски и слабый шум.
От этой точки дальше строятся все однофакторные эксперименты.


In [ ]:
base_scenario = research.make_scenario(
    'base_500x500_rank3',
    m=500,
    n=500,
    rank=3,
    observed_fraction=0.35,
    noise_std=0.02,
    missingness='random',
    description='Базовый сценарий: 500x500, rank=3, случайные пропуски, слабый шум.',
    tags=['base'],
)

{
    'name': base_scenario.name,
    'shape': (base_scenario.config.matrix.m, base_scenario.config.matrix.n),
    'rank': base_scenario.config.matrix.rank,
    'noise_std': base_scenario.config.noise.std,
    'observed_fraction': base_scenario.config.mask_sampling.observed_fraction,
    'missingness': base_scenario.config.missingness_field.mode,
}


## 2. Методы

Ранг метода по умолчанию берется из текущего сценария. Поэтому при переборе по рангу методы автоматически получают тот же ранг, что и матрица.


In [ ]:
methods = [
    research.make_method('soft_impute', 'Soft-Impute', max_iter=60),
    research.make_method('als', 'ALS', max_iter=60, init='spectral', reg=1e-3),
    research.make_method('compact_rgd', 'Compact RGD', max_iter=80, init='spectral'),
    research.make_method('compact_rgd_l2', 'Compact RGD + L2', max_iter=80, init='spectral', l2_reg=0.03),
]

[method.label for method in methods]


## 3. Однофакторные эксперименты

Если `values` не задан, сетка строится автоматически. Для ранга малые значения проверяются подряд, дальше шаг растет.
Ручной список тоже можно передать через `values=[...]`.


In [ ]:
sweeps = [
    research.make_sweep('rank_sweep', 'matrix.rank', alias='rank', title='Перебор по рангу'),
    research.make_sweep('noise_sweep', 'noise.std', alias='noise', title='Перебор по шуму'),
    research.make_sweep('missingness_sweep', 'missingness_field.mode', alias='missingness', title='Перебор по типу пропусков'),
]

default_grids = {sweep.alias: research.values_for_sweep(sweep, base_scenario) for sweep in sweeps}
default_grids


## 4. Запуск набора экспериментов

Один вызов `run_suite` запускает все однофакторные эксперименты подряд: сначала ранг, потом шум, потом тип пропусков.


In [ ]:
suite = research.make_suite(
    'core_research_suite',
    base_scenario,
    methods,
    sweeps,
    seeds=[41],
    description='Набор однофакторных экспериментов от одного базового сценария.',
    output_dir='research_outputs/core_research_suite',
)


In [ ]:
suite_run = research.run_suite(suite, PROJECT_DIR)

{
    'sweeps': len(suite_run.sweep_runs),
    'manifest_rows': len(suite_run.manifest),
    'records': len(suite_run.records),
    'summary_rows': len(suite_run.summary),
}


## 5. Перебор по рангу

Смотрим качество восстановления, время работы и число итераций.


In [ ]:
research.plot_sweep_report(
    suite_run,
    'rank_sweep',
    x='rank',
    title='Перебор по рангу',
    xlabel='Ранг',
    x_scale='log',
    max_xticks=7,
)


## 6. Перебор по шуму

Для шума используется симметричная логарифмическая шкала, чтобы дробные значения не слипались.


In [ ]:
research.plot_sweep_report(
    suite_run,
    'noise_sweep',
    x='noise',
    title='Перебор по шуму',
    xlabel='Шум',
    x_scale='symlog',
    x_tick_rotation=12,
    x_tick_sig_digits=2,
    max_xticks=5,
)


## 7. Перебор по типу пропусков

Категориальный эксперимент показывает, насколько методы устойчивы к структуре пропусков.


In [ ]:
research.plot_sweep_report(
    suite_run,
    'missingness_sweep',
    x='missingness',
    title='Перебор по типу пропусков',
    xlabel='Тип пропусков',
    x_tick_rotation=10,
)


## 8. Сохранение

Сохраняем базовый сценарий, конфиг набора экспериментов и таблицы результатов.


In [ ]:
saved_scenario = research.save_scenario(base_scenario, PROJECT_DIR)
saved_suite = research.save_suite(suite, PROJECT_DIR)
saved_run = research.save_run(suite_run)

{
    'scenario': saved_scenario,
    'suite': saved_suite,
    **saved_run,
}
